<a href="https://colab.research.google.com/github/eluwachinonso21-dev/Silicon-extraction-model/blob/main/Silicon_extraction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports and Library Installation**

In [ ]:
# Install required libraries
!pip -q install pandas numpy scikit-learn optuna cvxpy plotly

import pandas as pd
import numpy as np
import cvxpy as cp
import optuna
import plotly.express as px
import plotly.graph_objects as go
from google.colab import drive

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel

# Set seed for reproducibility
np.random.seed(42)

# **Load data**

In [ ]:

drive.mount('/content/drive')

# Load dataset
df = pd.read_csv("/content/drive/MyDrive/silicon project/silicon_extraction_data.csv")

# Clean numeric columns
cols_to_fix = ['Mg/SiO2','T (K)','T (min)','Mg2Si','Si','SiO2','MgO','Mg']
for col in cols_to_fix:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Handle design point metadata
df['is_design'] = (df['Note'].str.strip() == 'Des.').astype(int)
df = df.dropna(subset=cols_to_fix).reset_index(drop=True)

print(f"Data Cleaning Complete. Total Samples: {len(df)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data Cleaning Complete. Total Samples: 31


# **Physical Constants and Mass Balance**

In [ ]:
# Molar Masses (g/mol)
MW = {'Mg2Si': 76.71, 'Si': 28.08, 'SiO2': 60.08, 'MgO': 40.30, 'Mg': 24.30}
species_list = ['Mg2Si','Si','SiO2','MgO','Mg']
species_mws = np.array([MW[s] for s in species_list])

# Element Matrix (Atoms per molecule)
# Rows: Si, Mg, O | Columns: Mg2Si, Si, SiO2, MgO, Mg
A_molar = np.array([
    [1, 1, 1, 0, 0],  # Si balance
    [2, 0, 0, 1, 1],  # Mg balance
    [0, 0, 2, 1, 0]   # O balance
])

# Weight-basis matrix: A_wt_ij = (Atoms_i_in_j) / MW_j
A_wt = A_molar / species_mws

def get_b_init(mg_sio2_ratio):
    """Calculates initial atomic inventory based on feed ratio."""
    # Start with 1 mole SiO2 and R moles Mg
    n_init = np.array([0, 0, 1.0, 0, mg_sio2_ratio])
    return A_molar @ n_init

# **Modeling and Optimization**

In [ ]:
# Features and Targets
X = df[['Mg/SiO2', 'T (K)', 'T (min)', 'is_design']].values
Y = df[species_list].values

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

def project_to_physics(y_pred, b_init):
    """Quadratic programming to enforce mass balance and non-negativity."""
    y = cp.Variable(5)
    constraints = [y >= 0, cp.sum(y) == 100.0]
    Ay = A_wt @ y
    # Ratio constraints to match initial feed
    constraints += [Ay[0] * b_init[1] == Ay[1] * b_init[0]] # Si/Mg
    constraints += [Ay[2] * b_init[1] == Ay[1] * b_init[2]] # O/Mg

    prob = cp.Problem(cp.Minimize(cp.sum_squares(y - y_pred)), constraints)
    try:
        prob.solve(solver=cp.OSQP, verbose=False)
        return y.value if y.value is not None else np.clip(y_pred, 0, 100)
    except:
        return np.clip(y_pred, 0, 100)

def train_and_predict(params, X_tr, Y_tr, X_te, b_init_te):
    kernel = ConstantKernel(params['c']) * RBF(params['ls']) + WhiteKernel(params['noise'])
    models = []
    preds = []
    for i in range(5):
        m = GaussianProcessRegressor(kernel=kernel, alpha=params['alpha_reg'], normalize_y=True).fit(X_tr, Y_tr[:, i])
        models.append(m)
        preds.append(m.predict(X_te))

    Y_raw = np.column_stack(preds)
    Y_proj = np.array([project_to_physics(Y_raw[k], b_init_te[k]) for k in range(len(Y_raw))])
    return Y_proj, models

# Optuna Hyperparameter Tuning
def objective(trial):
    p = {
        'ls': trial.suggest_float('ls', 0.1, 20.0),
        'c': trial.suggest_float('c', 1e-2, 1e2, log=True),
        'noise': trial.suggest_float('noise', 1e-5, 1e-1, log=True),
        'alpha_reg': trial.suggest_float('alpha_reg', 1e-10, 1e-3, log=True)
    }
    kf = KFold(n_splits=3)
    scores = []
    for t_idx, v_idx in kf.split(X_train_s):
        b_v = np.array([get_b_init(r[0]) for r in X_train[v_idx]])
        Y_p, _ = train_and_predict(p, X_train_s[t_idx], Y_train[t_idx], X_train_s[v_idx], b_v)
        scores.append(r2_score(Y_train[v_idx], Y_p))
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

# Train Final Model
best_p = study.best_params
b_val_init = np.array([get_b_init(r[0]) for r in X_val])
Y_val_final, final_models = train_and_predict(best_p, X_train_s, Y_train, X_val_s, b_val_init)

print(f"\nFinal Validation R2: {r2_score(Y_val, Y_val_final):.4f}")

[I 2026-01-06 19:25:15,358] A new study created in memory with name: no-name-e7d78a02-3be4-471f-8df5-3eb2eb477d82
/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning:

The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.

/usr/local/lib/p


Final Validation R2: 0.8220


# **Visualization**

In [ ]:
# Create grid for visualization
T_grid = np.linspace(X[:,1].min(), X[:,1].max(), 30)
R_grid = np.linspace(X[:,0].min(), X[:,0].max(), 30)
T_mesh, R_mesh = np.meshgrid(T_grid, R_grid)

# Input for prediction (Fixed time at 60 mins)
X_surf = np.c_[R_mesh.ravel(), T_mesh.ravel(), np.full(T_mesh.size, 60), np.zeros(T_mesh.size)]
X_surf_s = scaler.transform(X_surf)
b_surf = np.array([get_b_init(r[0]) for r in X_surf])

# Predict
surf_raw = np.column_stack([m.predict(X_surf_s) for m in final_models])
surf_proj = np.array([project_to_physics(surf_raw[k], b_surf[k]) for k in range(len(surf_raw))])
Si_surf = surf_proj[:, 1].reshape(T_mesh.shape)

fig = go.Figure(data=[go.Surface(z=Si_surf, x=R_mesh, y=T_mesh, colorscale='Magma')])
fig.update_layout(title='Silicon Yield (wt%) Surface at 60 mins',
                  scene=dict(xaxis_title='Mg/SiO2 Ratio', yaxis_title='Temp (K)', zaxis_title='Si wt%'))
fig.show()

# **Interactive Predictor (Custom Input)**

In [ ]:
# --- USER INPUT SECTION ---
input_temp = 1073.15      # Kelvin
input_time = 360      # Minutes
input_ratio = 2.1     # Mg/SiO2
# --------------------------

def predict_custom(ratio, temp, time):
    x_cust = scaler.transform([[ratio, temp, time, 0]])
    b_cust = get_b_init(ratio)
    raw = np.array([m.predict(x_cust)[0] for m in final_models])
    proj = project_to_physics(raw, b_cust)

    print(f"\n--- Predicted Composition for {temp}K, {time}min, Ratio {ratio} ---")
    for name, val in zip(species_list, proj):
        print(f"{name}: {val:.2f} wt%")

    # Simple Bar Chart Output
    fig = px.bar(x=species_list, y=proj, labels={'x':'Species', 'y':'wt%'}, title="Predicted Species Distribution")
    fig.show()

predict_custom(input_ratio, input_temp, input_time)


--- Predicted Composition for 1073.15K, 360min, Ratio 2.1 ---
Mg2Si: 10.27 wt%
Si: 16.42 wt%
SiO2: 10.89 wt%
MgO: 57.92 wt%
Mg: 4.50 wt%
